# Phase 1: download each account's website

Reads the accounts in the input Excel and saves each website's pages (text and image labels) in the crawl store,
one line per account. Phase 2 reads only the crawl store.

**How it works** (`crawl.py`): the websites are split between several worker processes (`crawl_worker.py`), in up to
three passes: a fast plain download for every site, a real browser for the sites that refused or gave back next to
nothing, and a last try with an "undetected" browser (Chrome windows open) for those still refused. Each site's home
page comes first, then its most useful pages (`page_rules.py`), up to 40 pages (20 in the browser) and 150 seconds.
Each web address is crawled once, however many accounts have it.

CRM links to social media, directories, review and lead-generation sites, company-data sites and marketplaces are not
crawled (`accounts.LISTING_SITES`). A company's own Facebook or Instagram page is, narrowly: the page, its About page and
its own posts.

Stopping and re-running is always safe: saved accounts are skipped. Sites that refused us or didn't answer are
tried again after a short wait (up to 3 tries in a run), and again on every later run; only dead and parked sites
are dropped for good.

In [ ]:
# First time on a new machine only:
# !pip install -U crawl4ai pandas openpyxl requests model2vec python-dotenv
# !crawl4ai-setup

## Settings

In [ ]:
import crawl, datasets

SET = "crm_pilot"  # which accounts: "zoominfo_pilot", "crm_pilot", then "zoominfo_3", "crm_100" (datasets.py)
F = datasets.files(SET)
INPUT, STORE, SUMMARY = F.input, F.store, F.summary

LIMIT       = None  # crawl at most this many accounts this run (e.g. 5 to try it out); None = all that need it
RECRAWL_IDS = []    # account_ids to crawl again even though they are stored
REFRESH     = False  # True: read the accounts from the data warehouse again before making or topping up a set
RETRY_WAIT_MIN = 1  # sites that refused us or didn't answer: wait this long, then try them again (up to 3 tries);
                    #   None = don't wait, leave them for the next run
RECHECK_WEBSITE_ISSUES = False  # True: also crawl again the accounts whose website failed Phase 2's website
                                #   check ("website issue"); for an occasional sweep, not every run

## A new set of accounts (optional)
Add the set to `SETS` in `datasets.py` (input name, ZoomInfo or CRM, accounts per label, how to pick), set `SET`
above to it, and run the cell below once. The accounts come from the data warehouse (`data_warehouse.py`, the queries in
`queries/`): the latest snapshot in `data/snapshots`, or a fresh read with `REFRESH = True`. It picks per label (the
five segments, plus Other for the CRM), leaving out every account already crawled or in another set, and never
overwrites an existing set.

In [ ]:
# datasets.make(SET, refresh=REFRESH)  # writes the set's input file (once)

## Run

In [ ]:
crawl.run(INPUT, STORE, limit=LIMIT, recrawl_ids=RECRAWL_IDS, retry_wait_min=RETRY_WAIT_MIN,
          recheck_website_issues=RECHECK_WEBSITE_ISSUES)

## Short of usable accounts? (optional)
Counts the usable accounts per label (crawled fine) and, for any label short of the number wanted in `datasets.py`,
adds the next newest accounts to this set (the shortfall plus 20%). Then run the crawl cell above again: it crawls only
the new ones. `write=False` shows what it would add without changing anything.

In [ ]:
# datasets.top_up(SET, write=False)  # what it would add
# datasets.top_up(SET, refresh=REFRESH)  # add them, then re-run the crawl cell

## How the crawl went
| status | meaning |
|---|---|
| `ok` | pages saved; the reason says how far it got: `complete`, `partial` (time ran out), `thin` (one short page), `partial_bot_check` (a bot check let only some pages through: crawled again next run, gently, one page at a time) |
| `parked`, `dead_site` | the domain is for sale, or the site no longer exists (or the Facebook/Instagram page was deleted, or a website sends visitors to Facebook/Instagram but not to a company page). A placeholder ("coming soon"), a home page giving a 404 or a deleted Facebook/Instagram page is tried again on every run; a domain that doesn't exist and a parked domain are dropped |
| `blocked`, `unreachable` | it refused us (including a Facebook/Instagram log-in wall, also when a company website sends visitors to one) or didn't answer: tried again after RETRY_WAIT_MIN in the same run (up to 3 tries), and again on every later run |
| `network_blocked` | our own network filter stopped it |
| `no_website`, `social_media_page_only` | nothing to crawl: no link, or a social media or listing page (not a company's own Facebook/Instagram page) |
| `wrong_site` | set by Phase 2: a website issue (the site doesn't show this company); crawled again only when the CRM web address changes, or with `RECHECK_WEBSITE_ISSUES` |

Whatever the status, an account whose CRM web address has changed since its last crawl is crawled again.

In [ ]:
summary = crawl.summary(STORE, SUMMARY)
print(summary["status"].value_counts().to_string())
print("\nMedian pages per usable site:", summary.loc[summary.status == "ok", "pages"].median())
summary.head(20)

## Tidy the store (optional)
Re-crawls and website checks add lines; this keeps one line per account.

In [ ]:
# crawl.compact(STORE)